# Merchant Ranking Based on Essential-or-not

this notebook ranks merchants within three segments based on their category: 

- essential: needed in all circumstances 
- borderline: useful but leaning more discretionary 
- not-essential: other products 

In [1]:
import sys
from functools import reduce
from datetime import timedelta

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ExternalPreprocess").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 17:31:29 WARN Utils: Your hostname, Poppys-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.175 instead (on interface en0)
26/10/02 17:31:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/poppyvangerrevink/Documents/GitHub/industry-project-group-21/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/02 17:31:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where app

## Load Merged_df

In [ ]:
merged_df = spark.read.parquet("data/curated/merged_transactions_sa2.parquet")


root
 |-- postcode: integer (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- consumer_id: long (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- category_group: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_merchant_fraud_prob: double (nullable = true)
 |-- M

## Check Segments 

segments are based on company category tags.

In [5]:
merged_df.groupBy("is_essential").count().orderBy("is_essential").show()

+-------------+-----+
| is_essential|count|
+-------------+-----+
|   borderline|10623|
|    essential|11771|
|not_essential|49422|
+-------------+-----+



In [6]:
(merged_df
    .select("is_essential", "category")
    .distinct()
    .orderBy("is_essential", "category")
    .show(100, truncate=False))

+-------------+-------------------------------------------------------------------------------------+
|is_essential |category                                                                             |
+-------------+-------------------------------------------------------------------------------------+
|borderline   |cable, satellite, and other pay television and radio services                        |
|borderline   |computers, computer peripheral equipment, and software                               |
|borderline   |furniture, home furnishings and equipment shops, and manufacturers, except appliances|
|borderline   |stationery, office supplies and printing and writing paper                           |
|essential    |motor vehicle supplies and new parts                                                 |
|essential    |opticians, optical goods, and eyeglasses                                             |
|essential    |shoe shops                                                         

# Merchant Summary

In [14]:
# check nulls in merged_df
merged_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in merged_df.columns]).show()    
# check total rows in merged_df
merged_df.count()

def merchant_summary(spark_df):
    return spark_df.groupBy("merchant_abn").agg(
        F.first("merchant_name", ignorenulls=True).alias("merchant_name"),
        F.first("category", ignorenulls=True).alias("category"),
        F.first("is_essential", ignorenulls=True).alias("is_essential"),
        F.first("take_rate", ignorenulls=True).alias("take_rate"),
        F.sum("dollar_value").alias("total_revenue"),
        F.count("order_id").alias("num_transactions"),
        F.avg("dollar_value").alias("avg_order_value"),
        F.avg("fraud_probability").alias("mean_fraud_probability"),

    )

merchant_summary_df = merchant_summary(merged_df)
merchant_summary_df.show(10, truncate=False)
merchant_summary_df.groupBy("is_essential").count().orderBy("is_essential").show()

+--------+------------+-------+--------------+------------+--------+-----------+-----------------+---------------------+----+-------+-----+------+-------------+----+--------+------------+---------+--------------+-------------+--------------+---------------+-----------------------+------------------+-----------------------------+---------------------------+------------------+-------------------------+----------------------------+-------------------------+----------------------+----------+------------+
|postcode|merchant_abn|user_id|order_datetime|dollar_value|order_id|consumer_id|fraud_probability|is_same_day_duplicate|name|address|state|gender|merchant_name|tags|category|revenue_band|take_rate|category_group|total_revenue|n_transactions|avg_order_value|avg_merchant_fraud_prob|Median_age_persons|Median_mortgage_repay_monthly|Median_tot_prsnl_inc_weekly|Median_rent_weekly|Median_tot_fam_inc_weekly|Average_num_psns_per_bedroom|Median_tot_hhd_inc_weekly|Average_household_size|prop_18_34|is

# Ranking within Each Segment 

percentile ranking used for comparison on same scale
ranking score uses weighted average of percentile rankings 

In [20]:
FEATURES = {
    # score name:            (metric column,            higher_is_better, weight)
    "take_rate_score":       ("take_rate",              True,  1.0),
    "dollar_value_score":    ("total_dollar_value",     True,  1.0),
    "transaction_score":     ("num_transactions",       True,  1.0),
    "avg_order_value_score": ("avg_order_value",        True,  1.0),

}

In [21]:

def add_percentile_scores(summary_df):
    """Percentile score per feature within each segment (1 = best)."""
    out = summary_df
    for score_col, (metric, higher_is_better, _) in FEATURES.items():
        order = F.col(metric) if higher_is_better else F.desc(metric)
        w = Window.partitionBy("is_essential").orderBy(order)
        out = out.withColumn(score_col, F.percent_rank().over(w))
    return out


def add_ranking(scored_df):
    """Weighted average of the feature scores, ranked within each segment."""
    total_w = sum(wt for _, _, wt in FEATURES.values())
    score = sum(F.col(c) * wt for c, (_, _, wt) in FEATURES.items()) / total_w
    w = Window.partitionBy("is_essential").orderBy(
        F.desc("ranking_score"), F.asc("merchant_abn")  # merchant_abn breaks ties consistently
    )
    return (
        scored_df
        .withColumn("ranking_score", score)
        .withColumn("rank", F.row_number().over(w))
    )


ranking = add_ranking(add_percentile_scores(merchant_summary_df)).cache()

{"ts": "2026-10-02 18:15:28.705", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_dollar_value` cannot be resolved. Did you mean one of the following? [`avg_order_value`, `total_revenue`, `take_rate`, `category`, `is_essential`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor59.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1349.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_dollar_value` cannot be resolved. Did you mean one of the following? [`avg_order_value`, `total_revenue`, `take_rate`, `category`, `is_essential`]. SQLSTATE: 42703;\n'Project [merchant_abn#34, merchant_name#2609, category#2610

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_dollar_value` cannot be resolved. Did you mean one of the following? [`avg_order_value`, `total_revenue`, `take_rate`, `category`, `is_essential`]. SQLSTATE: 42703;
'Project [merchant_abn#34, merchant_name#2609, category#2610, is_essential#2611, take_rate#2612, total_revenue#2613, num_transactions#2614L, avg_order_value#2615, mean_fraud_probability#2616, take_rate_score#2771, percent_rank() windowspecdefinition(is_essential#2611, 'total_dollar_value ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS dollar_value_score#2781]
+- Project [merchant_abn#34, merchant_name#2609, category#2610, is_essential#2611, take_rate#2612, total_revenue#2613, num_transactions#2614L, avg_order_value#2615, mean_fraud_probability#2616, take_rate_score#2771]
   +- Project [merchant_abn#34, merchant_name#2609, category#2610, is_essential#2611, take_rate#2612,

In [15]:
SEGMENTS = {"essential", "borderline", "non-essential"}
for segment in SEGMENTS:
    print(f"Top merchants in {segment} segment:")
    ranking.filter(F.col("is_essential") == (segment == "essential")).orderBy(F.desc("ranking_score")).show(10, truncate=False)

Top merchants in borderline segment:


NameError: name 'ranking' is not defined